# Final-evaluation rerun: Logistic Regression

This notebook reruns the established tuning procedure on the new frozen development split created by notebook 10. It never loads the locked final-test rows. It selects validation thresholds for 70%, 75%, 80%, 85%, and 90% target recall, then saves the frozen model and artifacts for notebook 17.

## Data Preperation

In [1]:
import json
import joblib

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    fbeta_score,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import StratifiedKFold


In [3]:
# Locate the cleaned dataset, then redirect all final-rerun artifacts into a separate folder.

PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = parent / "Processed_Dataset" / "diabetic_data_cleaned_stage1.csv"
    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find Processed_Dataset/diabetic_data_cleaned_stage1.csv")

OUTPUT_DIR = PROJECT_ROOT / "Model_Results" / "prediction_model_optimisation_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print(DATA_PATH)
print(df.shape)
df.head()

# Final-evaluation artifacts are kept separate from the earlier development runs.
FINAL_EVALUATION_DIR = PROJECT_ROOT / "Final_Evaluation"
SPLIT_DIR = FINAL_EVALUATION_DIR / "Data_Splits"
OUTPUT_DIR = FINAL_EVALUATION_DIR / "Model_Artifacts" / "logistic_regression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nFinal-evaluation output directory:")
print(OUTPUT_DIR)


/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Processed_Dataset/diabetic_data_cleaned_stage1.csv
(69987, 56)

Final-evaluation output directory:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Final_Evaluation/Model_Artifacts/logistic_regression


In [4]:
# Reuse the same fixed 18 predictors as the other final model reruns.
target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

model_features = categorical_features + numeric_features

# Fail early if the cleaned dataset is missing any expected feature.
missing_features = [
    col for col in model_features
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"These modelling features are missing: {missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].astype(int).copy()


print("X shape:", X.shape)
print("Features used:", X.columns.tolist())
print("\nTarget distribution:")
print(y.value_counts())

# Guard against IDs, raw duplicates and the target leaking into the model input.
forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = forbidden_features.intersection(X.columns)

assert not unexpected_features, (
    f"Unexpected features found in X: {unexpected_features}"
)

assert not X.columns.duplicated().any(), (
    "Duplicate column names found in X."
)

print("Feature-selection checks passed.")


X shape: (69987, 18)
Features used: ['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target distribution:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64
Feature-selection checks passed.


In [5]:

# Load only the frozen model-training and threshold-validation rows.
# The locked final-test row file is intentionally not read in this notebook.

SPLIT_DIR = PROJECT_ROOT / "Final_Evaluation" / "Data_Splits"

train_split_path = SPLIT_DIR / "model_train_rows.csv"
validation_split_path = SPLIT_DIR / "threshold_validation_rows.csv"

if not train_split_path.exists() or not validation_split_path.exists():
    raise FileNotFoundError(
        "Final split files are missing. Run 10_create_final_split.ipynb first."
    )

train_idx = (
    pd.read_csv(train_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)
val_idx = (
    pd.read_csv(validation_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)

# Confirm that model-training and threshold-validation patients are disjoint.
assert set(train_idx).isdisjoint(set(val_idx))

X_model_train = X.iloc[train_idx].copy()
y_model_train = y.iloc[train_idx].copy()

X_val = X.iloc[val_idx].copy()
y_val = y.iloc[val_idx].copy()

split_summary = pd.DataFrame({
    "split": ["model_training", "threshold_validation"],
    "rows": [len(X_model_train), len(X_val)],
    "positive_count": [int(y_model_train.sum()), int(y_val.sum())],
    "positive_rate": [float(y_model_train.mean()), float(y_val.mean())],
})

print(
    "Final-test rows have NOT been loaded. "
    "They stay locked until notebook 17."
)
split_summary


Final-test rows have NOT been loaded. They stay locked until notebook 17.


,split,rows,positive_count,positive_rate
0,model_training,41991,3771,0.089805
1,threshold_validation,13998,1257,0.089799


In [6]:
# Categorical variables are imputed and one-hot encoded.
categorical_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

# Numerical variables are median-imputed and standardised for regularised Logistic Regression.
numeric_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

# Combine both preprocessing branches inside the model pipeline to prevent leakage.
preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            categorical_transformer,
            categorical_features
        ),
        (
            "num",
            numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)


In [7]:
def save_confusion_matrix(model, X_test, y_test, model_name, output_dir):
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual not readmitted", "Actual readmitted"],
        columns=["Predicted not readmitted", "Predicted readmitted"]
    )

    file_name = (
        model_name
        .lower()
        .replace(":", "")
        .replace(" ", "_")
        .replace("/", "_")
    )

    cm_df.to_csv(output_dir / f"confusion_matrix_{file_name}.csv")

    tn, fp, fn, tp = cm.ravel()

    total = tn + fp + fn + tp

    cm_long = pd.DataFrame([{
        "model": model_name,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "total": total,
        "true_negative_rate": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "false_positive_rate": fp / (tn + fp) if (tn + fp) > 0 else 0,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else 0,
        "true_positive_rate_recall": tp / (fn + tp) if (fn + tp) > 0 else 0
    }])

    return cm_df, cm_long

def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model"
):
    """
    Evaluate binary predictions created from predicted probabilities.

    Parameters
    ----------
    y_true:
        True labels, where 0 means not readmitted and
        1 means readmitted within 30 days.

    y_proba:
        Predicted probability of class 1.

    threshold:
        Probability threshold used to create class predictions.

    model_name:
        Name stored in the results table.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    # Convert predicted risk into a class label at the requested operating threshold.
    y_pred = (y_proba >= threshold).astype(int)

    # Extract TN/FP/FN/TP once for specificity, error rates and workload measures.
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )

    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )

    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    results = {
        "model": model_name,
        "threshold": float(threshold),

        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,

        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0
        ),

        # These probability-based metrics do not depend
        # on the classification threshold.
        "auroc": roc_auc_score(y_true, y_proba),
        "auprc": average_precision_score(
            y_true,
            y_proba
        ),
        "brier_score": brier_score_loss(
            y_true,
            y_proba
        ),

        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),

        "predicted_positive": int(predicted_positive),
        "predicted_negative": int(predicted_negative),
        "predicted_positive_rate": predicted_positive_rate,

        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        )
    }

    return results


def confusion_matrix_from_proba(
    y_true,
    y_proba,
    threshold=0.5
):
    """
    Create a labelled confusion matrix from probabilities.
    """

    y_pred = (np.asarray(y_proba) >= threshold).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    return pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted"
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted"
        ]
    )


def threshold_sweep(
    y_true,
    y_proba,
    model_name="Model",
    thresholds=None
):
    """
    Create a descriptive table showing performance
    across a range of probability thresholds.

    This table is useful for inspection and plotting.
    It is not used for final threshold selection.
    """

    if thresholds is None:
        thresholds = np.round(
            np.arange(0.05, 0.951, 0.01),
            2
        )

    results = [
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold,
            model_name=model_name
        )
        for threshold in thresholds
    ]

    return pd.DataFrame(results)


def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model"
):
    """
    Select the threshold with the lowest false-positive rate
    among thresholds achieving the required recall.

    Selection rule:
        1. Recall must be at least min_recall.
        2. Minimise the false-positive rate.
        3. If tied, use the highest threshold.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    # Evaluate all ROC thresholds rather than selecting from only a coarse manual grid.
    false_positive_rates, recalls, thresholds = roc_curve(
        y_true,
        y_proba,
        drop_intermediate=False
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": false_positive_rates,
        "specificity": 1 - false_positive_rates
    })

    # The first ROC threshold can be infinity.
    candidate_table = candidate_table[
        np.isfinite(candidate_table["threshold"])
    ].copy()

    # Keep only thresholds meeting the recall target, then choose the one with lowest FPR.
    eligible_candidates = candidate_table[
        candidate_table["recall"] >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            f"No threshold achieved recall >= {min_recall:.2f}."
        )

    eligible_candidates = eligible_candidates.sort_values(
        by=[
            "false_positive_rate",
            "threshold"
        ],
        ascending=[
            True,
            False
        ]
    ).reset_index(drop=True)

    selected_threshold = float(
        eligible_candidates.iloc[0]["threshold"]
    )

    selected_metrics = evaluate_predictions_from_proba(
        y_true=y_true,
        y_proba=y_proba,
        threshold=selected_threshold,
        model_name=model_name
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates
    )

## Regularised Logistic Regression optimisation

The model-training subset is used for hyperparameter tuning with
stratified cross-validation. Average precision, equivalent to AUPRC,
is used as the main cross-validation score because the outcome is
strongly imbalanced.

The validation set is then used to choose a classification threshold.
The threshold must achieve at least 80% recall for readmitted patients.
Among eligible thresholds, the threshold with the lowest false-positive
rate is selected.

The final test set is used once, after the model specification and
threshold have both been fixed.

In [8]:
# Hyperparameters are selected by 5-fold CV on model-training data.
# Probability thresholds are selected separately on the frozen validation set.
# Primary operating point used to compare/select the LR candidate.
PRIMARY_RECALL_TARGET = 0.80

# Additional operating points saved for later threshold-sensitivity analysis.
RECALL_TARGETS = [0.70, 0.75, 0.80, 0.85, 0.90]

# Keep the original variable name used by the existing selection cells below.
RECALL_TARGET = PRIMARY_RECALL_TARGET

# Stratification keeps the minority readmission rate similar across folds.
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

regularised_log_reg_param_grid = {
    # C spans strong to weak regularisation on a logarithmic scale.
    "model__C": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0
    ],

    # scikit-learn 1.8 uses l1_ratio instead of
    # the deprecated penalty parameter.
    #
    # 0.0 = pure L2
    # 1.0 = pure L1
    # values between them = Elastic Net
    # l1_ratio spans pure L2 through Elastic Net mixtures to pure L1.
    "model__l1_ratio": [
        0.0,
        0.25,
        0.5,
        0.75,
        1.0
    ],

    # Test whether class weighting improves ranking.
    # Test no weighting, automatic balance and progressively stronger positive-class weights.
    "model__class_weight": [
        None,
        "balanced",
        {0: 1, 1: 2},
        {0: 1, 1: 4},
        {0: 1, 1: 6},
        {0: 1, 1: 8},
        {0: 1, 1: 10},
    ]
}

In [9]:
# Candidate 1: additive Logistic Regression using the standard feature set.
additive_log_reg_pipeline = Pipeline(steps=[
    (
        "preprocess",
        preprocess
    ),
    (
        "model",
        LogisticRegression(
            solver="saga",
            max_iter=5000,
            tol=1e-3,
            random_state=42
        )
    )
])

# GridSearchCV evaluates every listed hyperparameter combination using mean CV AUPRC.
additive_log_reg_search = GridSearchCV(
    estimator=additive_log_reg_pipeline,
    param_grid=regularised_log_reg_param_grid,
    scoring="average_precision",
    cv=cross_validation,
    refit=True,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

# Fit the search using model-training rows only; validation rows are not used here.
additive_log_reg_search.fit(
    X_model_train,
    y_model_train
)

Fitting 5 folds for each of 175 candidates, totalling 875 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... tol=0.001))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.001, 0.01, ...], 'model__class_weight': [None, 'balanced', ...], 'model__l1_ratio': [0.0, 0.25, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denotin

In [10]:
print(
    "Best additive Logistic Regression parameters:"
)
print(additive_log_reg_search.best_params_)

print(
    "\nBest cross-validation AUPRC:"
)
print(additive_log_reg_search.best_score_)

additive_cv_results = pd.DataFrame(
    additive_log_reg_search.cv_results_
)

additive_cv_results_selected = (
    additive_cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "param_model__C",
            "param_model__l1_ratio",
            "param_model__class_weight"
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

additive_cv_results_selected.to_csv(
    OUTPUT_DIR
    / "regularised_logistic_regression_additive_cv_results.csv",
    index=False
)

additive_cv_results_selected.head(10)

Best additive Logistic Regression parameters:
{'model__C': 0.01, 'model__class_weight': {0: 1, 1: 2}, 'model__l1_ratio': 0.75}

Best cross-validation AUPRC:
0.14415059343017464


,rank_test_score,mean_test_score,std_test_score,mean_train_score,param_model__C,param_model__l1_ratio,param_model__class_weight
0,1,0.144151,0.005688,0.143613,0.010,0.75,"{0: 1, 1: 2}"
1,2,0.144124,0.005750,0.143509,0.010,0.50,None
2,3,0.144104,0.005749,0.143696,0.010,1.00,"{0: 1, 1: 4}"
3,4,0.144045,0.005696,0.143422,0.010,1.00,"{0: 1, 1: 2}"
4,5,0.144011,0.005687,0.144011,0.010,0.50,"{0: 1, 1: 2}"
5,6,0.143984,0.005662,0.144163,0.010,0.25,None
6,7,0.143965,0.005745,0.143492,0.010,1.00,balanced
7,8,0.143889,0.005822,0.143252,0.001,0.25,"{0: 1, 1: 8}"
8,9,0.143859,0.005795,0.143251,0.001,0.25,"{0: 1, 1: 10}"
9,10,0.143843,0.005632,0.144023,0.010,0.75,"{0: 1, 1: 4}"


In [11]:
# best_estimator_ is already refitted on all model-training rows with the winning settings.
additive_log_reg_model = (
    additive_log_reg_search.best_estimator_
)

y_val_proba_additive = (
    additive_log_reg_model
    .predict_proba(X_val)[:, 1]
)

In [12]:
# Compare the conventional 0.5 threshold with the recall-constrained validation threshold.
additive_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_additive,
        threshold=0.5,
        model_name=(
            "Regularised Logistic Regression "
            "additive validation default"
        )
    )
)

(
    additive_selected_threshold,
    additive_val_selected_results,
    additive_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_additive,
    min_recall=RECALL_TARGET,
    model_name=(
        "Regularised Logistic Regression "
        "additive validation selected"
    )
)

print(
    "Selected additive-model threshold:",
    additive_selected_threshold
)

pd.DataFrame([
    additive_val_default_results,
    additive_val_selected_results
])

Selected additive-model threshold: 0.12750359281229567


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Regularised Logistic Regression additive valid...,0.500000,0.909130,0.317073,0.010342,0.997802,0.002198,0.989658,0.020031,0.012823,...,0.148159,0.085772,12713,28,1244,13,41,13957,0.002929,3.153846
1,Regularised Logistic Regression additive valid...,0.127504,0.386984,0.107755,0.800318,0.346205,0.653795,0.199682,0.189937,0.350181,...,0.148159,0.085772,4411,8330,251,1006,9336,4662,0.666952,9.280318


In [13]:
additive_eligible_thresholds.head(10)

,threshold,recall,false_positive_rate,specificity
0,0.127504,0.800318,0.653795,0.346205
1,0.127497,0.800318,0.653873,0.346127
2,0.127480,0.800318,0.653952,0.346048
3,0.127478,0.800318,0.654109,0.345891
4,0.127448,0.800318,0.654187,0.345813
5,0.127447,0.800318,0.654266,0.345734
6,0.127447,0.800318,0.654344,0.345656
7,0.127446,0.800318,0.654423,0.345577
8,0.127443,0.800318,0.654580,0.345420
9,0.127443,0.800318,0.654658,0.345342


In [14]:
additive_threshold_sweep = threshold_sweep(
    y_true=y_val,
    y_proba=y_val_proba_additive,
    model_name=(
        "Regularised Logistic Regression additive validation"
    )
)

additive_threshold_sweep.to_csv(
    OUTPUT_DIR
    / "regularised_logistic_regression_additive_validation_threshold_sweep.csv",
    index=False
)

additive_threshold_sweep[
    [
        "threshold",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f2",
        "false_positive",
        "false_negative"
    ]
].sort_values(
    "threshold",
    ascending=False
)

,threshold,recall,precision,specificity,false_positive_rate,f2,false_positive,false_negative
90,0.95,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
89,0.94,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
88,0.93,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
87,0.92,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
86,0.91,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
...,...,...,...,...,...,...,...,...
4,0.09,0.998409,0.089868,0.002433,0.997567,0.330385,12710,2
3,0.08,1.000000,0.089799,0.000000,1.000000,0.330337,12741,0
2,0.07,1.000000,0.089799,0.000000,1.000000,0.330337,12741,0
1,0.06,1.000000,0.089799,0.000000,1.000000,0.330337,12741,0


In [15]:
additive_val_selected_cm = (
    confusion_matrix_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_additive,
        threshold=additive_selected_threshold
    )
)

additive_val_selected_cm.to_csv(
    OUTPUT_DIR
    / "confusion_matrix_regularised_logistic_regression_additive_validation.csv"
)

additive_val_selected_cm

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4411,8330
Actual readmitted,251,1006


In [16]:
additive_validation_comparison = pd.DataFrame([
    additive_val_default_results,
    additive_val_selected_results
])

additive_validation_comparison.to_csv(
    OUTPUT_DIR
    / "regularised_logistic_regression_additive_validation_results.csv",
    index=False
)

additive_validation_comparison

,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Regularised Logistic Regression additive valid...,0.500000,0.909130,0.317073,0.010342,0.997802,0.002198,0.989658,0.020031,0.012823,...,0.148159,0.085772,12713,28,1244,13,41,13957,0.002929,3.153846
1,Regularised Logistic Regression additive valid...,0.127504,0.386984,0.107755,0.800318,0.346205,0.653795,0.199682,0.189937,0.350181,...,0.148159,0.085772,4411,8330,251,1006,9336,4662,0.666952,9.280318


### Interaction experiment
HbA1c group × primary diagnosis interaction is tested below

In [17]:
INTERACTION_FEATURE = (
    "hba1c_x_primary_diagnosis"
)


def add_hba1c_diagnosis_interaction(X_data):
    """Create one categorical HbA1c × primary-diagnosis interaction feature.

    This allows the predictive Logistic Regression model to represent different
    HbA1c associations for different primary-diagnosis groups.
    """

    X_with_interaction = X_data.copy()

    hba1c_values = (
        X_with_interaction["hba1c_group"]
        .astype("string")
        .fillna("Missing")
    )

    diagnosis_values = (
        X_with_interaction["primary_diagnosis"]
        .astype("string")
        .fillna("Missing")
    )

    X_with_interaction[INTERACTION_FEATURE] = (
        hba1c_values
        + " | "
        + diagnosis_values
    )

    return X_with_interaction


X_model_train_interaction = (
    add_hba1c_diagnosis_interaction(
        X_model_train
    )
)

X_val_interaction = (
    add_hba1c_diagnosis_interaction(
        X_val
    )
)


interaction_categorical_features = (
    categorical_features
    + [INTERACTION_FEATURE]
)

print(
    X_model_train_interaction[
        [
            "hba1c_group",
            "primary_diagnosis",
            INTERACTION_FEATURE
        ]
    ].head()
)

                     hba1c_group primary_diagnosis  \
17838      No test was performed            Injury   
41870  Normal result of the test     Genitourinary   
15847      No test was performed       Circulatory   
15159      No test was performed       Circulatory   
37234      No test was performed       Circulatory   

                       hba1c_x_primary_diagnosis  
17838             No test was performed | Injury  
41870  Normal result of the test | Genitourinary  
15847        No test was performed | Circulatory  
15159        No test was performed | Circulatory  
37234        No test was performed | Circulatory  


In [18]:
interaction_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

interaction_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

interaction_preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            interaction_categorical_transformer,
            interaction_categorical_features
        ),
        (
            "num",
            interaction_numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)

In [19]:
# Candidate 2: repeat the same CV search after adding the explicit HbA1c × diagnosis interaction.
interaction_log_reg_pipeline = Pipeline(steps=[
    (
        "preprocess",
        interaction_preprocess
    ),
    (
        "model",
        LogisticRegression(
            solver="saga",
            max_iter=5000,
            tol=1e-3,
            random_state=42
        )
    )
])

interaction_log_reg_search = GridSearchCV(
    estimator=interaction_log_reg_pipeline,
    param_grid=regularised_log_reg_param_grid,
    scoring="average_precision",
    cv=cross_validation,
    refit=True,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

interaction_log_reg_search.fit(
    X_model_train_interaction,
    y_model_train
)

Fitting 5 folds for each of 175 candidates, totalling 875 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... tol=0.001))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.001, 0.01, ...], 'model__class_weight': [None, 'balanced', ...], 'model__l1_ratio': [0.0, 0.25, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denotin

In [20]:
print(
    "Best interaction Logistic Regression parameters:"
)
print(interaction_log_reg_search.best_params_)

print(
    "\nBest interaction cross-validation AUPRC:"
)
print(interaction_log_reg_search.best_score_)

interaction_cv_results = pd.DataFrame(
    interaction_log_reg_search.cv_results_
)

interaction_cv_results_selected = (
    interaction_cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "param_model__C",
            "param_model__l1_ratio",
            "param_model__class_weight"
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

interaction_cv_results_selected.to_csv(
    OUTPUT_DIR
    / "regularised_logistic_regression_interaction_cv_results.csv",
    index=False
)

interaction_cv_results_selected.head(10)

Best interaction Logistic Regression parameters:
{'model__C': 0.01, 'model__class_weight': 'balanced', 'model__l1_ratio': 1.0}

Best interaction cross-validation AUPRC:
0.14397559871621496


,rank_test_score,mean_test_score,std_test_score,mean_train_score,param_model__C,param_model__l1_ratio,param_model__class_weight
0,1,0.143976,0.005756,0.143504,0.010,1.00,balanced
1,2,0.143871,0.005778,0.143312,0.001,0.25,"{0: 1, 1: 8}"
2,3,0.143860,0.005759,0.143250,0.001,0.25,"{0: 1, 1: 10}"
3,4,0.143846,0.005762,0.143325,0.001,0.25,"{0: 1, 1: 6}"
4,5,0.143842,0.005718,0.144127,0.010,1.00,"{0: 1, 1: 6}"
5,6,0.143771,0.005383,0.144553,0.010,0.75,"{0: 1, 1: 6}"
6,7,0.143764,0.005729,0.143777,0.010,0.75,balanced
7,8,0.143743,0.006101,0.142418,0.001,1.00,"{0: 1, 1: 6}"
8,9,0.143743,0.006094,0.142418,0.001,0.75,"{0: 1, 1: 4}"
9,10,0.143686,0.005135,0.144920,0.010,0.50,"{0: 1, 1: 6}"


In [21]:
interaction_log_reg_model = (
    interaction_log_reg_search.best_estimator_
)

y_val_proba_interaction = (
    interaction_log_reg_model
    .predict_proba(X_val_interaction)[:, 1]
)

interaction_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_interaction,
        threshold=0.5,
        model_name=(
            "Regularised Logistic Regression "
            "interaction validation default"
        )
    )
)

(
    interaction_selected_threshold,
    interaction_val_selected_results,
    interaction_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_interaction,
    min_recall=RECALL_TARGET,
    model_name=(
        "Regularised Logistic Regression "
        "interaction validation selected"
    )
)

print(
    "Selected interaction-model threshold:",
    interaction_selected_threshold
)

pd.DataFrame([
    interaction_val_default_results,
    interaction_val_selected_results
])

Selected interaction-model threshold: 0.4225832283689996


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Regularised Logistic Regression interaction va...,0.500000,0.612016,0.128648,0.575179,0.615650,0.384350,0.424821,0.210266,0.339500,...,0.148408,0.237168,7844,4897,534,723,5620,8378,0.401486,7.773167
1,Regularised Logistic Regression interaction va...,0.422583,0.394699,0.109016,0.800318,0.354682,0.645318,0.199682,0.191893,0.352834,...,0.148408,0.237168,4519,8222,251,1006,9228,4770,0.659237,9.172962


In [22]:
# Compare the two Logistic Regression variants at their own validation-selected thresholds.
logistic_regression_validation_comparison = pd.DataFrame([
    additive_val_selected_results,
    interaction_val_selected_results
])

comparison_columns = [
    "model",
    "threshold",
    "auprc",
    "auroc",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "f2",
    "true_positive",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found"
]

logistic_regression_validation_comparison = (
    logistic_regression_validation_comparison[
        comparison_columns
    ]
)

logistic_regression_validation_comparison.to_csv(
    OUTPUT_DIR
    / "logistic_regression_candidate_validation_comparison.csv",
    index=False
)

logistic_regression_validation_comparison

,model,threshold,auprc,auroc,recall,precision,specificity,false_positive_rate,f2,true_positive,false_positive,false_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Regularised Logistic Regression additive valid...,0.127504,0.148159,0.628478,0.800318,0.107755,0.346205,0.653795,0.350181,1006,8330,251,0.666952,9.280318
1,Regularised Logistic Regression interaction va...,0.422583,0.148408,0.630108,0.800318,0.109016,0.354682,0.645318,0.352834,1006,8222,251,0.659237,9.172962


In [23]:
# Candidate selection first enforces the recall target, then prioritises lower FPR,
# higher precision and higher AUPRC.
eligible_logistic_models = (
    logistic_regression_validation_comparison[
        logistic_regression_validation_comparison[
            "recall"
        ] >= RECALL_TARGET
    ]
    .copy()
)

if eligible_logistic_models.empty:
    raise ValueError(
        "No Logistic Regression candidate achieved "
        f"validation recall >= {RECALL_TARGET:.2f}."
    )

selected_logistic_row = (
    eligible_logistic_models
    .sort_values(
        by=[
            "false_positive_rate",
            "precision",
            "auprc"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .iloc[0]
)

selected_logistic_name = (
    selected_logistic_row["model"]
)

print(
    "Selected Logistic Regression candidate:"
)
print(selected_logistic_name)

selected_logistic_row

Selected Logistic Regression candidate:
Regularised Logistic Regression interaction validation selected


model                                          Regularised Logistic Regression interaction va...
threshold                                                                               0.422583
auprc                                                                                   0.148408
auroc                                                                                   0.630108
recall                                                                                  0.800318
precision                                                                               0.109016
specificity                                                                             0.354682
false_positive_rate                                                                     0.645318
f2                                                                                      0.352834
true_positive                                                                               1006
false_positive                

## Freeze final development-stage artifacts

In [24]:
# Freeze the selected Logistic Regression variant and save several recall operating points.
# These artifacts are consumed later without reopening model selection.
if "additive" in selected_logistic_name.lower():
    final_logistic_model = additive_log_reg_model
    final_logistic_val_proba = y_val_proba_additive
    final_logistic_variant = "additive"
else:
    final_logistic_model = interaction_log_reg_model
    final_logistic_val_proba = y_val_proba_interaction
    final_logistic_variant = "interaction"


threshold_rows = []

# Save 70–90% recall operating points for later sensitivity comparison.
for recall_target in RECALL_TARGETS:
    selected_threshold, selected_metrics, _ = (
        choose_threshold_for_minimum_recall(
            y_true=y_val,
            y_proba=final_logistic_val_proba,
            min_recall=recall_target,
            model_name="Regularised Logistic Regression validation"
        )
    )

    threshold_rows.append({
        "target_recall": recall_target,
        "selected_threshold": selected_threshold,
        "validation_recall": selected_metrics["recall"],
        "validation_precision": selected_metrics["precision"],
        "validation_specificity": selected_metrics["specificity"],
        "validation_false_positive_rate": selected_metrics["false_positive_rate"],
        "validation_f2": selected_metrics["f2"],
        "validation_true_positive": selected_metrics["true_positive"],
        "validation_false_negative": selected_metrics["false_negative"],
        "validation_false_positive": selected_metrics["false_positive"],
        "validation_true_negative": selected_metrics["true_negative"],
        "validation_flagged_rate": selected_metrics["predicted_positive_rate"],
    })

selected_thresholds = pd.DataFrame(threshold_rows)

selected_thresholds.to_csv(
    OUTPUT_DIR / "selected_validation_thresholds.csv",
    index=False
)

validation_predictions = pd.DataFrame({
    "row_position": val_idx,
    "y_true": np.asarray(y_val, dtype=int),
    "probability": np.asarray(final_logistic_val_proba, dtype=float),
})

validation_predictions.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

selected_thresholds

joblib.dump(
    final_logistic_model,
    OUTPUT_DIR / "final_model.joblib"
)

with open(OUTPUT_DIR / "model_variant.json", "w") as f:
    json.dump(
        {"variant": final_logistic_variant},
        f,
        indent=2
    )

with open(OUTPUT_DIR / "selected_hyperparameters.json", "w") as f:
    json.dump(
        {
            "variant": final_logistic_variant,
            "additive_best_params": additive_log_reg_search.best_params_,
            "interaction_best_params": interaction_log_reg_search.best_params_,
        },
        f,
        indent=2,
        default=str
    )

print("Saved frozen Logistic Regression model and artifacts.")
print("Selected variant:", final_logistic_variant)


Saved frozen Logistic Regression model and artifacts.
Selected variant: interaction
